In [4]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [5]:
# Load document 

loader = PyPDFLoader("Alchemist.pdf")

docs = loader.load()

In [7]:
len(docs)

136

In [18]:
# text splitter split docs into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300,separators=["\n\n", "\n", " ", ""])

chunks = text_splitter.split_documents(docs)

In [19]:
len(chunks)

342

In [16]:
# model defining for embedding 
embedding_model = OllamaEmbeddings(model="qwen3-embedding:0.6b")

In [17]:
llm_model = ChatOllama(model="deepseek-r1:1.5b")

In [20]:
# vector store 

vector_store = Chroma.from_documents(chunks,embedding=embedding_model)

In [21]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [23]:
user_query = """how paulo coelho discover his life meaning?"""

In [24]:
retriever_result = retriever.invoke(user_query)

In [30]:
retriever_result

[Document(id='ab5e26c7-381c-40fa-8f30-f183d1652cdc', metadata={'source': 'Alchemist.pdf', 'page': 5}, page_content='bring not only him but others who read this fine book closer to\nrecognizing and reaching their own inner destinies.”\n—Charlotte Zolotow , author of If You Listen\n“Paulo Coelho gives you the inspiration to follow your own dreams\nby seeing the world through your own eyes and not someone else’ s.”\n—Lynn Andrews, author of the Medicine W oman series\n“Nothing is impossible, such is Coelho’ s message, as long as you\nwish it with all your heart. No other book bears so much hope; small\nwonder its author became a guru among all those in search of the\nmeaning of life.”\n—Focus  (Germany)\n“The Alchemist  is a truly poetic book.”\n—Welt am Sonntag  (Germany)'),
 Document(id='62ecf8fe-bcfc-4eee-9674-16456453513f', metadata={'page': 130, 'source': 'Alchemist.pdf'}, page_content='achieved a self-awareness and a spiritual awakening that he later\ndescribed in The Pilgrimage.\nP

In [25]:
prompt = f"""
    You are a helpful assistant that helps users find information about the book "The Alchemist" by Paulo Coelho. Use the following context to answer the question at the end.
    Context: {retriever_result}
    Question: {user_query}
"""

In [26]:
result = llm_model.invoke([HumanMessage(content=prompt)])

In [29]:
print(result.content)

Paulo Coelho discovered his life meaning through several key elements of his approach:

1. **Inspirational Books**: His book "Companion" presents philosophical stories with a warrior theme, encouraging readers to embrace their personal destinities through inspiration and purpose.

2. **Personal Journey and Motivation**: Despite facing numerous challenges—such as exploring various professions and enduring prison—he continued writing, indicating internal motivation to continue his career and personal growth.

3. **Award and Recognition**: Receiving prestigious awards like the UN Messenger of Peace highlights recognition for his humanitarian efforts and selfless contributions to society.

4. **Personal Identity and Success**: His life story shows he didn't give up on writing despite external pressures; this perseverance reflects a connection between his inner journey and his professional work.

Together, these elements emphasize that Paulo Coelho's meaning is deeply intertwined with his p